In [1]:
# ==============================================================================
# NOTEBOOK: Bowler-Model-Training.ipynb
# DESCRIPTION: Training Bowler Model (Wickets) with Validation Split & Pipelines
# ==============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

# Set global random state
RANDOM_STATE = 42

# ===========================================================
# STAGE 1: LOAD AND ALIGN DATA
# ===========================================================
print("="*60)
print("STAGE 1: LOADING & ALIGNING BOWLER DATA")
print("="*60)

bowler = pd.read_csv('bowler_features_final.csv')
bowler['date'] = pd.to_datetime(bowler['date'])

# Target: Wickets in THIS match
y = bowler['wickets']

# Features to DROP (Leakage & Identifiers)
drop_cols = [
    'wickets', 'runs_conceded', 'economy', 'balls_bowled', 'overs_bowled',
    'target_wickets', 'target_runs_conceded', 'target_economy',
    'match_id', 'player', 'date', 'next_opponent', 'next_venue'
]
X = bowler.drop(columns=drop_cols)

print(f"Features Shape: {X.shape}")



STAGE 1: LOADING & ALIGNING BOWLER DATA
Features Shape: (11542, 43)


In [2]:
# ===========================================================
# STAGE 2: TRAIN - VALIDATION - TEST SPLIT (Strict Time Series)
# ===========================================================
print("\n" + "="*60)
print("STAGE 2: TIME-SERIES SPLIT")
print("="*60)

# Test: 2024 onwards
test_mask = bowler['date'] >= '2024-01-01'
X_test = X[test_mask].drop(columns=['season'])
y_test = y[test_mask]

# Validation: 2023 (for XGBoost stopping)
val_mask = (bowler['date'] >= '2023-01-01') & (bowler['date'] < '2024-01-01')
X_val = X[val_mask].drop(columns=['season'])
y_val = y[val_mask]

# Train: 2008 to 2022
train_mask = bowler['date'] < '2023-01-01'
X_train = X[train_mask].drop(columns=['season'])
y_train = y[train_mask]

print(f"Training Set   (2008-2022): {X_train.shape}")
print(f"Validation Set (2023 Only): {X_val.shape}")
print(f"Testing Set    (2024-2025): {X_test.shape}")




STAGE 2: TIME-SERIES SPLIT
Training Set   (2008-2022): (9508, 42)
Validation Set (2023 Only): (714, 42)
Testing Set    (2024-2025): (1320, 42)


In [3]:
# ===========================================================
# STAGE 3: DEFINE PREPROCESSING PIPELINE
# ===========================================================
cat_cols = ['team', 'opponent', 'venue', 'city', 'innings', 'experience_level']
num_cols = [col for col in X_train.columns if col not in cat_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ]
)

# ===========================================================
# STAGE 4: BASELINE COMPARISON
# ===========================================================
print("\n" + "="*60)
print("STAGE 4: BASELINE COMPARISON")
print("="*60)

# 1. Mean Baseline
mean_pred = np.full(len(y_test), y_train.mean())
mae_mean = mean_absolute_error(y_test, mean_pred)

# 2. Smart Baseline (Recent Form: wickets_last_10)
# If a bowler takes 2 wickets usually, predicting 2 is a good guess.
smart_pred = X_test['wickets_last_10'].fillna(y_train.mean())
mae_smart = mean_absolute_error(y_test, smart_pred)

print(f"📉 Dumb Baseline (Global Mean): MAE = {mae_mean:.3f}")
print(f"📉 Smart Baseline (Last 10 Avg): MAE = {mae_smart:.3f}")




STAGE 4: BASELINE COMPARISON
📉 Dumb Baseline (Global Mean): MAE = 0.816
📉 Smart Baseline (Last 10 Avg): MAE = 0.879


In [4]:
# ===========================================================
# STAGE 5: MODEL TRAINING
# ===========================================================
print("\n" + "="*60)
print("STAGE 5: MODEL TRAINING")
print("="*60)

# --- Random Forest Pipeline ---
print("🌲 Training Random Forest Pipeline...")
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        n_estimators=200, max_depth=10, n_jobs=-1, random_state=RANDOM_STATE
    ))
])

rf_pipeline.fit(X_train, y_train)
y_pred_rf = rf_pipeline.predict(X_test)
mae_rf = mean_absolute_error(y_test, y_pred_rf)
print(f"✅ Random Forest MAE: {mae_rf:.3f}")

# --- XGBoost (Manual Preprocessing for Eval Set) ---
print("\n🚀 Training XGBoost...")
X_train_proc = preprocessor.fit_transform(X_train)
X_val_proc = preprocessor.transform(X_val)
X_test_proc = preprocessor.transform(X_test)

xgb_model = xgb.XGBRegressor(
    n_estimators=1000, learning_rate=0.01, max_depth=5,
    early_stopping_rounds=50, n_jobs=-1, random_state=RANDOM_STATE
)

xgb_model.fit(
    X_train_proc, y_train,
    eval_set=[(X_val_proc, y_val)],
    verbose=False
)

y_pred_xgb = xgb_model.predict(X_test_proc)
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
print(f"✅ XGBoost MAE: {mae_xgb:.3f}")




STAGE 5: MODEL TRAINING
🌲 Training Random Forest Pipeline...
✅ Random Forest MAE: 0.115

🚀 Training XGBoost...
✅ XGBoost MAE: 0.121


In [5]:
# ===========================================================
# STAGE 6: FINAL SAVE
# ===========================================================
print("\n" + "="*60)
print("STAGE 6: SAVING FINAL ARTIFACTS")
print("="*60)

# Save Pipeline (Safest option)
joblib.dump(rf_pipeline, 'bowler_pipeline.pkl')
print("✅ Saved: bowler_pipeline.pkl")

# Save XGBoost separately
joblib.dump(xgb_model, 'bowler_xgb.pkl')
joblib.dump(preprocessor, 'bowler_preprocessor.pkl')

results_df = pd.DataFrame({
    'Model': ['Mean Baseline', 'Smart Baseline', 'Random Forest', 'XGBoost'],
    'MAE': [mae_mean, mae_smart, mae_rf, mae_xgb]
})
print("\nFinal Leaderboard:")
print(results_df.sort_values('MAE'))


STAGE 6: SAVING FINAL ARTIFACTS
✅ Saved: bowler_pipeline.pkl

Final Leaderboard:
            Model       MAE
2   Random Forest  0.115204
3         XGBoost  0.120674
0   Mean Baseline  0.816284
1  Smart Baseline  0.879422
